# Market Regime Detection for Portfolio Allocation with RL + Explainability

This notebook builds a regime-aware portfolio allocation pipeline and then explains policy behavior with temporal attention-style diagnostics.

## What This Notebook Covers
1. Regime detection with Gaussian HMM
2. RL training with DQN, A2C, and PPO
3. Test-period evaluation and policy behavior diagnostics
4. Explainability views:
   - DQN surrogate attention + gradient saliency
   - Finance dashboard linking returns, actions, and regimes
   - PPO post-hoc surrogate attention + policy saliency

## Important Clarification
- DQN section uses value-based learning and is explained with surrogate attention + saliency.
- PPO section in this notebook uses `MlpPolicy` (no native attention layer in PPO policy).
- PPO attention plots are post-hoc explainability, not internal PPO attention weights.

## Execution Guide (Recommended Order)

1. Run Sections 1 to 5 to prepare data and environments.
2. Choose training path:
   - DQN path: Section 6 + Sections 7 to 9.2
   - PPO path: PPO training cell + Section 9.3
3. Run Section 10 only when comparing against the baseline setup.

This order keeps variables aligned and avoids stale-state confusion in long notebook sessions.

## 1. Install and Import Dependencies

In [1]:
import os
import sys
from pathlib import Path

repo_root = Path.cwd().resolve()
if not (repo_root / "ml").exists():
    found_root = None
    for candidate in [repo_root] + list(repo_root.parents):
        if (candidate / "ml").exists() and (candidate / "data").exists():
            found_root = candidate
            break
    if found_root is None:
        raise FileNotFoundError("Could not locate project root containing 'ml' and 'data'.")
    repo_root = found_root

sys.path.insert(0, str(repo_root))

def _rel_display(path_obj: Path) -> str:
    try:
        return str(path_obj.relative_to(repo_root))
    except ValueError:
        return str(path_obj)

import numpy as np
import pandas as pd
import seaborn as sns
import warnings

# FinRL (stable-baselines3)
from stable_baselines3 import DQN, A2C, PPO
from stable_baselines3.common.vec_env import DummyVecEnv

# ML libraries
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

# Project modules
from ml.models import GaussianHMMRegimeDetector
from ml.environments import WeeklyPortfolioEnv
from ml.training_utils import train_dqn_finrl, evaluate_episode
from ml import load_hyperparameter_config

# Visualization
from tqdm import tqdm

warnings.filterwarnings('ignore')

# Load centralized hyperparameters
hp_bundle = load_hyperparameter_config()
hp_cfg = hp_bundle.values
FAST_MODE = hp_bundle.fast_mode

env_cfg = hp_cfg['environment']
dqn_cfg = hp_cfg['dqn']
a2c_cfg = hp_cfg['a2c']
ppo_cfg = hp_cfg['ppo']
ppo_attn_cfg = hp_cfg['ppo_native_attention']
hpo_cfg = hp_cfg['hpo']
SEQ_LEN = hp_cfg['general']['sequence_length']

print(f"Hyperparameter mode: {'FAST' if FAST_MODE else 'FULL'}")
print(f"Configured sequence length: {SEQ_LEN}")

# Saved model artifacts for split-notebook workflow
model_dir = repo_root / 'output' / 'models'
model_dir.mkdir(parents=True, exist_ok=True)
MODEL_PATHS = {
    'dqn': model_dir / 'dqn_finrl_agent.zip',
    'a2c': model_dir / 'a2c_finrl_agent.zip',
    'ppo': model_dir / 'ppo_finrl_agent.zip',
    'ppo_native_attention': model_dir / 'ppo_native_attention_finrl_agent.zip',
}
print(f"Model artifacts directory: {_rel_display(model_dir)}")

# Set random seeds for reproducibility
np.random.seed(42)
import torch
torch.manual_seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed(42)
elif torch.backends.mps.is_available():
    torch.mps.manual_seed(42)

# Device configuration (order: CUDA > MPS > CPU)
if torch.cuda.is_available():
    device = "cuda"
elif torch.backends.mps.is_available():
    device = "mps"
else:
    device = "cpu"
print(f"Using device: {device}")

sns.set_palette("husl")

Hyperparameter mode: FULL
Configured sequence length: 4
Model artifacts directory: output/models
Using device: cuda


## 2. Load and Prepare Data from Pipeline

In [2]:
# Load processed data from the data pipeline
# Use repo_root so this works regardless of current working directory.
data_path = repo_root / 'data' / 'processed'

# Load weekly features (market + macro)
market_features = pd.read_csv(data_path / 'market_features_weekly.csv', index_col=0, parse_dates=True)
macro_features = pd.read_csv(data_path / 'macro_features_weekly.csv', index_col=0, parse_dates=True)
asset_targets = pd.read_csv(data_path / 'weekly_asset_targets.csv', index_col=0, parse_dates=True)

print("Data path: data/processed")
print(f"Market features shape: {market_features.shape}")
print(f"Macro features shape: {macro_features.shape}")
print(f"Asset targets shape: {asset_targets.shape}")

# Align indices
common_index = market_features.index.intersection(macro_features.index).intersection(asset_targets.index)
market_features = market_features.loc[common_index].sort_index()
macro_features = macro_features.loc[common_index].sort_index()
asset_targets = asset_targets.loc[common_index].sort_index()

print(f"\nAligned data shape: {market_features.shape}")
print(f"Date range: {market_features.index[0]} to {market_features.index[-1]}")

# Combine features for regime detection
all_features = pd.concat([market_features, macro_features], axis=1)
print(f"Combined features for HMM: {all_features.shape}")
print(f"Features: {list(all_features.columns)[:10]}...")

Data path: data/processed
Market features shape: (638, 37)
Macro features shape: (638, 33)
Asset targets shape: (638, 8)

Aligned data shape: (638, 37)
Date range: 2014-01-03 00:00:00 to 2026-03-20 00:00:00
Combined features for HMM: (638, 70)
Features: ['spy_ret_1d', 'spy_ret_5d', 'spy_ret_20d', 'spy_vol_5d', 'spy_vol_20d', 'spy_drawdown_60d', 'spy_ma_gap_5_20', 'spy_intraday_range', 'spy_volume_z_20', 'tlt_ret_1d']...


## 3. Gaussian HMM Regime Detection

In [3]:
# Split into Train/Validation/Test (2014-2020 / 2021-2022 / 2023-2026)
split_date_train = '2020-12-31'
split_date_val = '2022-12-31'

train_mask = all_features.index <= split_date_train
val_mask = (all_features.index > split_date_train) & (all_features.index <= split_date_val)
test_mask = all_features.index > split_date_val

train_features = all_features[train_mask].reset_index(drop=True)
train_returns = asset_targets[train_mask]

val_features = all_features[val_mask].reset_index(drop=True)
val_returns = asset_targets[val_mask]

test_features = all_features[test_mask].reset_index(drop=True)
test_returns = asset_targets[test_mask]

# Select only numeric columns and drop NaN
numeric_cols = train_features.select_dtypes(include=[np.number]).columns
train_features = train_features[numeric_cols].fillna(0)
val_features = val_features[numeric_cols].fillna(0)
test_features = test_features[numeric_cols].fillna(0)

print(f"Train data: {len(train_features)} weeks")
print(f"Val data: {len(val_features)} weeks")
print(f"Test data: {len(test_features)} weeks")
print(f"Train features shape: {train_features.shape} (numeric features only)")

# Fit HMM on training data
print("\n=== Fit Gaussian HMM ===")
hmm = GaussianHMMRegimeDetector(n_regimes=4, pca_components=10, random_state=42)
hmm.fit(train_features, regime_names=["Risk-On", "Neutral", "Defensive", "Panic"])

print(f"HMM fitted with {hmm.n_regimes} regimes")
print(f"Regime names: {hmm.get_regime_names()}")

# Predict regimes for all periods
train_regime_posteriors = hmm.predict_proba(train_features)
val_regime_posteriors = hmm.predict_proba(val_features)
test_regime_posteriors = hmm.predict_proba(test_features)

print(f"\nRegime posteriors shape:")
print(f"  Train: {train_regime_posteriors.shape}")
print(f"  Val: {val_regime_posteriors.shape}")
print(f"  Test: {test_regime_posteriors.shape}")

# Visualize regime detection with Plotly
import plotly.graph_objects as go
from plotly.subplots import make_subplots

predicted_regimes = hmm.predict_regimes(train_features)
fig = make_subplots(
    rows=2,
    cols=1,
    shared_xaxes=False,
    vertical_spacing=0.12,
    subplot_titles=(
        'Market Regimes (Train Period) - HMM Posterior Probabilities',
        'Most Likely Market Regime Over Time',
    ),
)

for regime_id in range(hmm.n_regimes):
    fig.add_trace(
        go.Scatter(
            x=list(range(len(train_regime_posteriors))),
            y=train_regime_posteriors[:, regime_id],
            mode='lines',
            name=hmm.regime_names[regime_id],
        ),
        row=1,
        col=1,
    )

fig.add_trace(
    go.Scatter(
        x=list(range(len(predicted_regimes))),
        y=predicted_regimes,
        mode='markers',
        marker=dict(size=6, opacity=0.7),
        name='Predicted Regime',
        showlegend=False,
    ),
    row=2,
    col=1,
)

fig.update_xaxes(title_text='Week', row=1, col=1)
fig.update_yaxes(title_text='Probability', row=1, col=1)
fig.update_xaxes(title_text='Week', row=2, col=1)
fig.update_yaxes(
    title_text='Regime ID',
    tickmode='array',
    tickvals=list(range(hmm.n_regimes)),
    ticktext=hmm.regime_names,
    row=2,
    col=1,
)
fig.update_layout(height=820, width=1250, legend=dict(orientation='h', y=1.02, x=0))
fig.show()

print("\nRegime Statistics (Train Period):")
regime_stats = hmm.get_regime_stats()
for regime_id, stats in regime_stats.items():
    print(f"\n{hmm.regime_names[regime_id]}:")
    for feature, value in list(stats.items())[:5]:
        print(f"  {feature}: {value:.4f}")

Train data: 365 weeks
Val data: 105 weeks
Test data: 168 weeks
Train features shape: (365, 66) (numeric features only)

=== Fit Gaussian HMM ===
HMM fitted with 4 regimes
Regime names: ['Risk-On', 'Neutral', 'Defensive', 'Panic']

Regime posteriors shape:
  Train: (365, 4)
  Val: (105, 4)
  Test: (168, 4)



Regime Statistics (Train Period):

Risk-On:
  spy_ret_1d: -0.0003
  spy_ret_5d: 0.0020
  spy_ret_20d: 0.0085
  spy_vol_5d: 0.0076
  spy_vol_20d: 0.0081

Neutral:
  spy_ret_1d: 0.0006
  spy_ret_5d: 0.0029
  spy_ret_20d: 0.0114
  spy_vol_5d: 0.0064
  spy_vol_20d: 0.0070

Defensive:
  spy_ret_1d: 0.0031
  spy_ret_5d: -0.0192
  spy_ret_20d: -0.0840
  spy_vol_5d: 0.0399
  spy_vol_20d: 0.0407

Panic:
  spy_ret_1d: 0.0019
  spy_ret_5d: 0.0100
  spy_ret_20d: 0.0471
  spy_vol_5d: 0.0127
  spy_vol_20d: 0.0150


## 4. Create Training, Validation, and Test Environments

In [4]:
# Asset returns for portfolio: [SPY, TLT, GLD, Cash(~0)]
# Check available columns and use those
print(f"Available columns in asset_targets: {asset_targets.columns.tolist()}")

# Use columns that match the expected assets
spy_col = 'SPY_return' if 'SPY_return' in asset_targets.columns else next((c for c in asset_targets.columns if 'SPY' in c.upper()), None)
tlt_col = 'TLT_return' if 'TLT_return' in asset_targets.columns else next((c for c in asset_targets.columns if 'TLT' in c.upper()), None)
gld_col = 'GLD_return' if 'GLD_return' in asset_targets.columns else next((c for c in asset_targets.columns if 'GLD' in c.upper()), None)

print(f"Using columns: SPY={spy_col}, TLT={tlt_col}, GLD={gld_col}")

spy_returns = asset_targets[spy_col] if spy_col else np.random.randn(len(asset_targets)) * 0.02
tlt_returns = asset_targets[tlt_col] if tlt_col else np.random.randn(len(asset_targets)) * 0.01
gld_returns = asset_targets[gld_col] if gld_col else np.random.randn(len(asset_targets)) * 0.015
cash_returns = np.zeros_like(spy_returns)  # Cash has ~0 return

portfolio_returns = pd.DataFrame({
    'SPY': spy_returns.values if hasattr(spy_returns, 'values') else spy_returns,
    'TLT': tlt_returns.values if hasattr(tlt_returns, 'values') else tlt_returns,
    'GLD': gld_returns.values if hasattr(gld_returns, 'values') else gld_returns,
    'Cash': cash_returns,
})

# Normalize features (important for neural network)
scaler = StandardScaler()
train_market_scaled = scaler.fit_transform(train_features)
val_market_scaled = scaler.transform(val_features)
test_market_scaled = scaler.transform(test_features)

train_features_normalized = train_features.copy()
train_features_normalized[:] = train_market_scaled

val_features_normalized = val_features.copy()
val_features_normalized[:] = val_market_scaled

test_features_normalized = test_features.copy()
test_features_normalized[:] = test_market_scaled

# Create environments from centralized config
train_env = WeeklyPortfolioEnv(
    features=train_features_normalized,
    regime_posteriors=train_regime_posteriors,
    asset_returns=portfolio_returns.iloc[:len(train_features_normalized)],
    transaction_cost=env_cfg['transaction_cost'],
    volatility_penalty=env_cfg['volatility_penalty'],
    lookback_vol=env_cfg['lookback_vol'],
    seq_len=SEQ_LEN
)

val_env = WeeklyPortfolioEnv(
    features=val_features_normalized,
    regime_posteriors=val_regime_posteriors,
    asset_returns=portfolio_returns.iloc[len(train_features_normalized):len(train_features_normalized)+len(val_features_normalized)],
    transaction_cost=env_cfg['transaction_cost'],
    volatility_penalty=env_cfg['volatility_penalty'],
    lookback_vol=env_cfg['lookback_vol'],
    seq_len=SEQ_LEN
)

test_env = WeeklyPortfolioEnv(
    features=test_features_normalized,
    regime_posteriors=test_regime_posteriors,
    asset_returns=portfolio_returns.iloc[len(train_features_normalized)+len(val_features_normalized):],
    transaction_cost=env_cfg['transaction_cost'],
    volatility_penalty=env_cfg['volatility_penalty'],
    lookback_vol=env_cfg['lookback_vol'],
    seq_len=SEQ_LEN
)

print(f"Environment Configuration:")
print(f"  State dimension: {train_env.observation_space.shape}")
print(f"  Action dimension: {train_env.action_space.n}")
print(f"  Sequence length: {train_env.seq_len}")
print(f"  Portfolio templates: {len(train_env.PORTFOLIO_TEMPLATES)}")
print(f"\nPortfolio Actions:")
for action_id, name in enumerate(train_env.ACTION_NAMES):
    print(f"  {action_id}: {name}")

Available columns in asset_targets: ['spy_weekly_close', 'tlt_weekly_close', 'gld_weekly_close', 'week_last_trade_date', 'next_return_spy', 'next_return_tlt', 'next_return_gld', 'source']
Using columns: SPY=spy_weekly_close, TLT=tlt_weekly_close, GLD=gld_weekly_close
Environment Configuration:
  State dimension: (4, 74)
  Action dimension: 7
  Sequence length: 4
  Portfolio templates: 7

Portfolio Actions:
  0: cash_only
  1: spy_only
  2: tlt_only
  3: gld_only
  4: spy_80_tlt_20
  5: balanced_60_30_10
  6: defensive_20_60_20


## 5. Initialize Attention DQN Agent

In [5]:
# Initialize DQN agent using centralized hyperparameter config
agent = DQN(
    'MlpPolicy',
    train_env,
    learning_rate=dqn_cfg['learning_rate'],
    buffer_size=dqn_cfg['buffer_size'],
    learning_starts=dqn_cfg['learning_starts'],
    batch_size=dqn_cfg['batch_size'],
    tau=1.0,
    gamma=dqn_cfg['gamma'],
    train_freq=1,
    target_update_interval=dqn_cfg['target_update_interval'],
    exploration_fraction=dqn_cfg['exploration_fraction'],
    exploration_initial_eps=dqn_cfg['exploration_initial_eps'],
    exploration_final_eps=dqn_cfg['exploration_final_eps'],
    device=device,
    verbose=1
)

print("=== FinRL DQN Agent Initialized ===")
print(f"Device: {agent.device}")
print(f"Policy: MlpPolicy")
print(f"State space: {train_env.observation_space}")
print(f"Action space: {train_env.action_space}")
print(f"Learning rate: {agent.learning_rate}")
print(f"Gamma: {agent.gamma}")
print(f"\nBuffer size: {agent.buffer_size}")
print(f"Batch size: {agent.batch_size}")
print(f"Target update interval: {agent.target_update_interval}")

# Test forward pass (gymnasium returns (obs, info) tuple)
obs, info = train_env.reset()
print(f"\nObservation shape: {obs.shape}")
print(f"Info keys: {info.keys() if isinstance(info, dict) else 'N/A'}")

Using cuda device
Wrapping the env with a `Monitor` wrapper
Wrapping the env in a DummyVecEnv.
=== FinRL DQN Agent Initialized ===
Device: cuda
Policy: MlpPolicy
State space: Box(-inf, inf, (4, 74), float32)
Action space: Discrete(7)
Learning rate: 0.0001
Gamma: 0.99

Buffer size: 250000
Batch size: 256
Target update interval: 2000

Observation shape: (4, 74)
Info keys: dict_keys([])


## 6. Train Attention-Enhanced DQN Agent

In [6]:
# Train the FinRL DQN agent using centralized config (progress bar only)
import importlib
import ml.training_utils as training_utils_mod
training_utils_mod = importlib.reload(training_utils_mod)

training_results = training_utils_mod.train_dqn_finrl(
    train_env=train_env,
    val_env=val_env,
    total_timesteps=dqn_cfg['train_timesteps'],
    eval_freq=dqn_cfg['eval_freq'],
    early_stopping_patience=dqn_cfg['early_stopping_patience'],
    learning_rate=dqn_cfg['learning_rate'],
    exploration_fraction=dqn_cfg['exploration_fraction'],
    exploration_final_eps=dqn_cfg['exploration_final_eps'],
    target_update_interval=dqn_cfg['target_update_interval'],
    buffer_size=dqn_cfg['buffer_size'],
    batch_size=dqn_cfg['batch_size'],
    device=device,
    verbose=0,
    callback_verbose=0,
 )

agent = training_results['agent']
val_history = training_results['val_history']

Output()

In [7]:
# Plot DQN validation-reward curve from training history (Plotly)
if 'val_history' not in globals() or not val_history:
    print('No validation history found yet. Run the DQN training cell first.')
else:
    import plotly.graph_objects as go

    val_steps = [h.get('step', i) for i, h in enumerate(val_history)]
    val_rewards = [h.get('reward', np.nan) for h in val_history]

    fig = go.Figure()
    fig.add_trace(
        go.Scatter(
            x=val_steps,
            y=val_rewards,
            mode='lines+markers',
            name='Validation reward',
        )
    )

    # Add a smooth trend line when enough points are available.
    if len(val_rewards) >= 5:
        trend = pd.Series(val_rewards).rolling(window=5, min_periods=1).mean()
        fig.add_trace(
            go.Scatter(
                x=val_steps,
                y=trend,
                mode='lines',
                name='Rolling mean (window=5)',
                line=dict(width=3),
            )
        )

    best_idx = int(np.nanargmax(val_rewards)) if len(val_rewards) else 0
    fig.add_trace(
        go.Scatter(
            x=[val_steps[best_idx]],
            y=[val_rewards[best_idx]],
            mode='markers',
            name='Best validation point',
            marker=dict(size=11),
        )
    )

    fig.update_layout(
        title='DQN Training Progress: Validation Reward vs Timesteps',
        xaxis_title='Timesteps',
        yaxis_title='Validation reward',
        height=460,
        width=1200,
        legend=dict(orientation='h', y=1.02, x=0),
    )
    fig.show()

    print(f"Validation checkpoints: {len(val_history)}")
    print(f"Best validation reward: {val_rewards[best_idx]:.4f} at step {val_steps[best_idx]}")

Validation checkpoints: 60
Best validation reward: 38779.9355 at step 5000


In [8]:
# Save DQN weights
if 'agent' in globals():
    MODEL_PATHS['dqn'].parent.mkdir(parents=True, exist_ok=True)
    agent.save(str(MODEL_PATHS['dqn']))
    print(f"Saved DQN weights to: {MODEL_PATHS['dqn']}")
else:
    print('DQN agent not found. Run the DQN training cell first.')

Saved DQN weights to: /home/yuaylong/Market-Regime-Detection-for-RL-Allocation/output/models/dqn_finrl_agent.zip


## 6.1 Train Actor-Critic (A2C) + Action Probability Dynamics

In [9]:
# Train an Actor-Critic policy (A2C) and visualize action probabilities over time
print("=== Training Actor-Critic (A2C) ===")
print(f"FAST_MODE={FAST_MODE} | train_steps={a2c_cfg['train_timesteps']} | n_steps={a2c_cfg['n_steps']}")

import plotly.express as px

# A2C is a policy-gradient actor-critic model with explicit action probabilities.
ac_agent = A2C(
    'MlpPolicy',
    train_env,
    learning_rate=a2c_cfg['learning_rate'],
    gamma=a2c_cfg['gamma'],
    n_steps=a2c_cfg['n_steps'],
    ent_coef=a2c_cfg['ent_coef'],
    vf_coef=a2c_cfg['vf_coef'],
    device=device,
    verbose=0,
)

ac_total_timesteps = a2c_cfg['train_timesteps']
ac_agent.learn(total_timesteps=ac_total_timesteps, progress_bar=not FAST_MODE)
print(f"A2C training complete. Timesteps: {ac_total_timesteps}")

# Evaluate deterministic actor-critic policy on test period
ac_test_eval = evaluate_episode(ac_agent, test_env, deterministic=True)
print("\nActor-Critic Test Performance:")
print(f"  Total Reward: {ac_test_eval['reward']:.4f}")
print(f"  Cumulative Return: {ac_test_eval.get('cumulative_return', 0):.4f}")
print(f"  Sharpe Ratio: {ac_test_eval.get('sharpe_ratio', 0):.4f}")
print(f"  Max Drawdown: {ac_test_eval.get('max_drawdown', 0):.4f}")

# Roll forward and record per-step action probability vector
obs, _ = test_env.reset()
prob_rows = []
max_steps_prob = min(200, len(test_env.features))
action_names_local = list(train_env.ACTION_NAMES)

for t in range(max_steps_prob):
    obs_tensor = torch.tensor(obs, dtype=torch.float32, device=ac_agent.device).unsqueeze(0)
    with torch.no_grad():
        dist = ac_agent.policy.get_distribution(obs_tensor)
        probs = dist.distribution.probs.squeeze(0).detach().cpu().numpy()

    action, _ = ac_agent.predict(obs, deterministic=True)
    action_id = int(action.item() if isinstance(action, np.ndarray) else action)

    row = {'step': t, 'chosen_action': action_names_local[action_id]}
    for i, name in enumerate(action_names_local):
        row[name] = float(probs[i])
    prob_rows.append(row)

    obs, reward, terminated, truncated, _ = test_env.step(action_id)
    if terminated or truncated:
        break

prob_df = pd.DataFrame(prob_rows)
prob_long = prob_df.melt(
    id_vars=['step', 'chosen_action'],
    value_vars=action_names_local,
    var_name='action_name',
    value_name='probability',
)

fig_probs = px.line(
    prob_long,
    x='step',
    y='probability',
    color='action_name',
    title='Actor-Critic: Action Probabilities Over Time',
    labels={'probability': 'Action probability', 'step': 'Test step', 'action_name': 'Action'},
)
fig_probs.update_layout(height=520, width=1200, legend=dict(orientation='h', y=1.02, x=0))
fig_probs.show()

fig_chosen = px.scatter(
    prob_df,
    x='step',
    y='chosen_action',
    title='Actor-Critic: Chosen Action Timeline',
    labels={'step': 'Test step', 'chosen_action': 'Chosen action'},
)
fig_chosen.update_layout(height=360, width=1200)
fig_chosen.show()

print("Action probability diagnostics ready.")

Output()

=== Training Actor-Critic (A2C) ===
FAST_MODE=False | train_steps=60000 | n_steps=256


A2C training complete. Timesteps: 60000

Actor-Critic Test Performance:
  Total Reward: 66509.2075
  Cumulative Return: inf
  Sharpe Ratio: 24.8716
  Max Drawdown: nan


Action probability diagnostics ready.


In [10]:
# Save A2C weights
if 'ac_agent' in globals():
    MODEL_PATHS['a2c'].parent.mkdir(parents=True, exist_ok=True)
    ac_agent.save(str(MODEL_PATHS['a2c']))
    print(f"Saved A2C weights to: {MODEL_PATHS['a2c']}")
else:
    print('A2C agent not found. Run the A2C training cell first.')

Saved A2C weights to: /home/yuaylong/Market-Regime-Detection-for-RL-Allocation/output/models/a2c_finrl_agent.zip


## 6.2 Train PPO + Action Probability Dynamics

In [11]:
# Train PPO and visualize action probabilities over time (config-driven)
print("=== Training PPO ===")

import plotly.express as px

ppo_train_steps = ppo_cfg['train_timesteps']
ppo_diag_steps = ppo_cfg['diagnostics_steps']
print(f"FAST_MODE={FAST_MODE} | train_steps={ppo_train_steps} | diag_steps={ppo_diag_steps}")

ppo_agent = PPO(
    'MlpPolicy',
    train_env,
    learning_rate=ppo_cfg['learning_rate'],
    n_steps=ppo_cfg['n_steps'],
    batch_size=ppo_cfg['batch_size'],
    n_epochs=ppo_cfg['n_epochs'],
    gamma=ppo_cfg['gamma'],
    gae_lambda=ppo_cfg['gae_lambda'],
    clip_range=ppo_cfg['clip_range'],
    ent_coef=ppo_cfg['ent_coef'],
    vf_coef=ppo_cfg['vf_coef'],
    device=device,
    verbose=0,
)

ppo_total_timesteps = ppo_train_steps
ppo_agent.learn(total_timesteps=ppo_total_timesteps, progress_bar=not FAST_MODE)
print(f"PPO training complete. Timesteps: {ppo_total_timesteps}")

ppo_test_eval = evaluate_episode(ppo_agent, test_env, deterministic=True)
print("\nPPO Test Performance:")
print(f"  Total Reward: {ppo_test_eval['reward']:.4f}")
print(f"  Cumulative Return: {ppo_test_eval.get('cumulative_return', 0):.4f}")
print(f"  Sharpe Ratio: {ppo_test_eval.get('sharpe_ratio', 0):.4f}")
print(f"  Max Drawdown: {ppo_test_eval.get('max_drawdown', 0):.4f}")

obs, _ = test_env.reset()
ppo_prob_rows = []
max_steps_prob = min(ppo_diag_steps, len(test_env.features))
action_names_local = list(train_env.ACTION_NAMES)

for t in range(max_steps_prob):
    obs_tensor = torch.tensor(obs, dtype=torch.float32, device=ppo_agent.device).unsqueeze(0)
    with torch.no_grad():
        ppo_dist = ppo_agent.policy.get_distribution(obs_tensor)
        ppo_probs = ppo_dist.distribution.probs.squeeze(0).detach().cpu().numpy()

    action, _ = ppo_agent.predict(obs, deterministic=True)
    action_id = int(action.item() if isinstance(action, np.ndarray) else action)

    row = {'step': t, 'chosen_action': action_names_local[action_id]}
    for i, name in enumerate(action_names_local):
        row[name] = float(ppo_probs[i])
    ppo_prob_rows.append(row)

    obs, reward, terminated, truncated, _ = test_env.step(action_id)
    if terminated or truncated:
        break

ppo_prob_df = pd.DataFrame(ppo_prob_rows)
ppo_prob_long = ppo_prob_df.melt(
    id_vars=['step', 'chosen_action'],
    value_vars=action_names_local,
    var_name='action_name',
    value_name='probability',
)

fig_ppo_probs = px.line(
    ppo_prob_long,
    x='step',
    y='probability',
    color='action_name',
    title='PPO: Action Probabilities Over Time',
    labels={'probability': 'Action probability', 'step': 'Test step', 'action_name': 'Action'},
)
fig_ppo_probs.update_layout(height=520, width=1200, legend=dict(orientation='h', y=1.02, x=0))
fig_ppo_probs.show()

fig_ppo_chosen = px.scatter(
    ppo_prob_df,
    x='step',
    y='chosen_action',
    title='PPO: Chosen Action Timeline',
    labels={'step': 'Test step', 'chosen_action': 'Chosen action'},
)
fig_ppo_chosen.update_layout(height=360, width=1200)
fig_ppo_chosen.show()

print("PPO action probability diagnostics ready.")

Output()

=== Training PPO ===
FAST_MODE=False | train_steps=120000 | diag_steps=300


PPO training complete. Timesteps: 120000

PPO Test Performance:
  Total Reward: 85742.2416
  Cumulative Return: inf
  Sharpe Ratio: 41.0782
  Max Drawdown: nan


PPO action probability diagnostics ready.


In [12]:
# Save PPO weights
if 'ppo_agent' in globals():
    MODEL_PATHS['ppo'].parent.mkdir(parents=True, exist_ok=True)
    ppo_agent.save(str(MODEL_PATHS['ppo']))
    print(f"Saved PPO weights to: {MODEL_PATHS['ppo']}")
else:
    print('PPO agent not found. Run the PPO training cell first.')

Saved PPO weights to: /home/yuaylong/Market-Regime-Detection-for-RL-Allocation/output/models/ppo_finrl_agent.zip


## 6.3 Train PPO with Native Temporal Attention Policy

> This section makes PPO use an internal attention feature extractor (native attention in the policy network), not only post-hoc explainability.

In [13]:
# Native-attention PPO: custom temporal attention feature extractor for policy/value networks
print("=== Training PPO with Native Temporal Attention ===")

import torch.nn as nn
from gymnasium import spaces
from stable_baselines3.common.torch_layers import BaseFeaturesExtractor
import plotly.express as px

ppo_attn_train_steps = ppo_attn_cfg['train_timesteps']
ppo_attn_diag_steps = ppo_attn_cfg['diagnostics_steps']
print(f"FAST_MODE={FAST_MODE} | train_steps={ppo_attn_train_steps} | diag_steps={ppo_attn_diag_steps}")

class TemporalAttentionExtractor(BaseFeaturesExtractor):
    """Apply self-attention over the temporal axis of [T, F] observations."""
    def __init__(self, observation_space: spaces.Box, features_dim: int = 128):
        super().__init__(observation_space, features_dim)
        if len(observation_space.shape) != 2:
            raise ValueError(
                f"Expected observation shape [T, F], got {observation_space.shape}. "
                "Set env to return sequence windows before using this extractor."
            )

        feat_dim = observation_space.shape[1]
        d_model = 64
        n_heads = 4

        self.input_proj = nn.Linear(feat_dim, d_model)
        self.self_attn = nn.MultiheadAttention(d_model, n_heads, batch_first=True)
        self.norm = nn.LayerNorm(d_model)
        self.mlp = nn.Sequential(
            nn.Linear(d_model, d_model),
            nn.ReLU(),
            nn.Linear(d_model, features_dim),
            nn.ReLU(),
        )

        self.latest_attn_weights = None

    def forward(self, observations: torch.Tensor) -> torch.Tensor:
        x = self.input_proj(observations)
        attn_out, attn_weights = self.self_attn(x, x, x, need_weights=True, average_attn_weights=False)
        x = self.norm(x + attn_out)
        pooled = x.mean(dim=1)
        self.latest_attn_weights = attn_weights.detach()
        return self.mlp(pooled)

def collect_policy_probabilities(policy_agent, env, action_names, max_steps=200):
    obs, _ = env.reset()
    rows = []
    max_steps = min(max_steps, len(env.features))

    for t in range(max_steps):
        obs_tensor = torch.tensor(obs, dtype=torch.float32, device=policy_agent.device).unsqueeze(0)
        with torch.no_grad():
            dist = policy_agent.policy.get_distribution(obs_tensor)
            probs = dist.distribution.probs.squeeze(0).detach().cpu().numpy()

        action, _ = policy_agent.predict(obs, deterministic=True)
        action_id = int(action.item() if isinstance(action, np.ndarray) else action)

        row = {'step': t, 'chosen_action': action_names[action_id]}
        for i, name in enumerate(action_names):
            row[name] = float(probs[i])
        rows.append(row)

        obs, _, terminated, truncated, _ = env.step(action_id)
        if terminated or truncated:
            break

    prob_df = pd.DataFrame(rows)
    prob_long = prob_df.melt(
        id_vars=['step', 'chosen_action'],
        value_vars=action_names,
        var_name='action_name',
        value_name='probability',
    )
    return prob_df, prob_long

def plot_action_diagnostics(prob_df, prob_long, title_prefix):
    fig_probs = px.line(
        prob_long,
        x='step',
        y='probability',
        color='action_name',
        title=f'{title_prefix}: Action Probabilities Over Time',
        labels={'probability': 'Action probability', 'step': 'Test step', 'action_name': 'Action'},
    )
    fig_probs.update_layout(height=520, width=1200, legend=dict(orientation='h', y=1.02, x=0))
    fig_probs.show()

    fig_chosen = px.scatter(
        prob_df,
        x='step',
        y='chosen_action',
        title=f'{title_prefix}: Chosen Action Timeline',
        labels={'step': 'Test step', 'chosen_action': 'Chosen action'},
    )
    fig_chosen.update_layout(height=360, width=1200)
    fig_chosen.show()

policy_kwargs_attn = dict(
    features_extractor_class=TemporalAttentionExtractor,
    features_extractor_kwargs=dict(features_dim=ppo_attn_cfg['features_dim']),
    net_arch=ppo_attn_cfg['net_arch'],
)

ppo_attn_agent = PPO(
    'MlpPolicy',
    train_env,
    learning_rate=ppo_attn_cfg['learning_rate'],
    n_steps=ppo_attn_cfg['n_steps'],
    batch_size=ppo_attn_cfg['batch_size'],
    n_epochs=ppo_attn_cfg['n_epochs'],
    gamma=ppo_attn_cfg['gamma'],
    gae_lambda=ppo_attn_cfg['gae_lambda'],
    clip_range=ppo_attn_cfg['clip_range'],
    ent_coef=ppo_attn_cfg['ent_coef'],
    vf_coef=ppo_attn_cfg['vf_coef'],
    policy_kwargs=policy_kwargs_attn,
    device=device,
    verbose=0,
    )

ppo_attn_total_timesteps = ppo_attn_train_steps
ppo_attn_agent.learn(total_timesteps=ppo_attn_total_timesteps, progress_bar=not FAST_MODE)
print(f"Native-attention PPO training complete. Timesteps: {ppo_attn_total_timesteps}")

ppo_attn_test_eval = evaluate_episode(ppo_attn_agent, test_env, deterministic=True)
print("\nNative-Attention PPO Test Performance:")
print(f"  Total Reward: {ppo_attn_test_eval['reward']:.4f}")
print(f"  Cumulative Return: {ppo_attn_test_eval.get('cumulative_return', 0):.4f}")
print(f"  Sharpe Ratio: {ppo_attn_test_eval.get('sharpe_ratio', 0):.4f}")
print(f"  Max Drawdown: {ppo_attn_test_eval.get('max_drawdown', 0):.4f}")

action_names_local = list(train_env.ACTION_NAMES)
attn_prob_df, attn_prob_long = collect_policy_probabilities(
    policy_agent=ppo_attn_agent,
    env=test_env,
    action_names=action_names_local,
    max_steps=ppo_attn_diag_steps,
    )
plot_action_diagnostics(attn_prob_df, attn_prob_long, 'Native-Attention PPO')

obs, _ = test_env.reset()
obs_batch = torch.tensor(obs, dtype=torch.float32, device=ppo_attn_agent.device).unsqueeze(0)
_ = ppo_attn_agent.policy.extract_features(obs_batch)
extractor = ppo_attn_agent.policy.features_extractor
native_attn = getattr(extractor, 'latest_attn_weights', None)

if native_attn is not None:
    native_attn_mean = native_attn.mean(dim=(0, 1)).detach().cpu().numpy()
    token_labels = [f"t-{train_env.seq_len-1-i}" for i in range(train_env.seq_len)]
    fig_native_attn = px.imshow(
        native_attn_mean,
        x=token_labels,
        y=token_labels,
        color_continuous_scale='Teal',
        title='Native PPO Attention Matrix (mean over heads)',
        labels={'x': 'Key timestep', 'y': 'Query timestep', 'color': 'weight'},
    )
    fig_native_attn.update_layout(width=760, height=620)
    fig_native_attn.show()
else:
    print("Native attention weights not available yet. Run one predict/eval step and try again.")

print("Native-attention PPO diagnostics ready.")

Output()

=== Training PPO with Native Temporal Attention ===
FAST_MODE=False | train_steps=140000 | diag_steps=300


Native-attention PPO training complete. Timesteps: 140000

Native-Attention PPO Test Performance:
  Total Reward: 86325.3338
  Cumulative Return: inf
  Sharpe Ratio: 41.4247
  Max Drawdown: nan


Native-attention PPO diagnostics ready.


In [14]:
# Save PPO Native Attention weights
if 'ppo_attn_agent' in globals():
    MODEL_PATHS['ppo_native_attention'].parent.mkdir(parents=True, exist_ok=True)
    ppo_attn_agent.save(str(MODEL_PATHS['ppo_native_attention']))
    print(f"Saved PPO Native Attention weights to: {MODEL_PATHS['ppo_native_attention']}")
else:
    print('PPO Native Attention agent not found. Run the native-attention PPO training cell first.')

Saved PPO Native Attention weights to: /home/yuaylong/Market-Regime-Detection-for-RL-Allocation/output/models/ppo_native_attention_finrl_agent.zip


## 6.4 Model Comparison: DQN vs PPO-MLP vs PPO-Native-Attention

This section compares test performance and action diversity across the three main agents.

Metrics: total reward, cumulative return, Sharpe ratio, max drawdown, and action entropy (diversity).

In [15]:
# Compare DQN, PPO (MLP), and PPO (Native Attention) in one panel
print("=== Model Comparison: DQN vs PPO-MLP vs PPO-Native-Attention ===")

import plotly.express as px
import plotly.graph_objects as go

def _safe_metric(eval_dict, key):
    return float(eval_dict.get(key, np.nan)) if isinstance(eval_dict, dict) else np.nan

def _action_entropy_from_series(action_series):
    if action_series is None or len(action_series) == 0:
        return np.nan
    probs = action_series.value_counts(normalize=True).values
    return float(-(probs * np.log(probs + 1e-12)).sum())

def _build_model_row(model_name, eval_dict=None, action_df=None, action_col=None):
    entropy = np.nan
    if isinstance(action_df, pd.DataFrame) and action_col in action_df.columns:
        entropy = _action_entropy_from_series(action_df[action_col])
    elif isinstance(eval_dict, dict) and 'actions' in eval_dict:
        tmp_df = pd.DataFrame(eval_dict['actions'])
        if 'action_name' in tmp_df.columns:
            entropy = _action_entropy_from_series(tmp_df['action_name'])

    return {
        'model': model_name,
        'reward': _safe_metric(eval_dict, 'reward'),
        'cumulative_return': _safe_metric(eval_dict, 'cumulative_return'),
        'sharpe_ratio': _safe_metric(eval_dict, 'sharpe_ratio'),
        'max_drawdown': _safe_metric(eval_dict, 'max_drawdown'),
        'action_entropy': entropy,
    }

rows = [
    _build_model_row('DQN', globals().get('test_eval'), globals().get('actions_df'), 'action_name'),
    _build_model_row('PPO-MLP', globals().get('ppo_test_eval'), globals().get('ppo_prob_df'), 'chosen_action'),
    _build_model_row('PPO-Native-Attn', globals().get('ppo_attn_test_eval'), globals().get('attn_prob_df'), 'chosen_action'),
]

cmp_df = pd.DataFrame(rows)
print(cmp_df.round(4))

metric_cols = ['reward', 'cumulative_return', 'sharpe_ratio', 'max_drawdown', 'action_entropy']
if cmp_df[metric_cols].isna().all(axis=None):
    raise RuntimeError(
        "No model results found. Run: DQN eval (Section 8), PPO-MLP (Section 6.2), and/or PPO-Native-Attn (Section 6.3)."
    )

perf_long = cmp_df.melt(
    id_vars='model',
    value_vars=['reward', 'cumulative_return', 'sharpe_ratio', 'max_drawdown'],
    var_name='metric',
    value_name='value',
)

fig_perf = px.bar(
    perf_long,
    x='metric',
    y='value',
    color='model',
    barmode='group',
    title='Performance Comparison: DQN vs PPO Variants',
    labels={'metric': 'Metric', 'value': 'Value', 'model': 'Model'},
)
fig_perf.update_layout(height=520, width=1280, legend=dict(orientation='h', y=1.02, x=0))
fig_perf.show()

fig_entropy = px.bar(
    cmp_df,
    x='model',
    y='action_entropy',
    color='model',
    title='Action Diversity (Entropy) Comparison',
    labels={'action_entropy': 'Action entropy', 'model': 'Model'},
)
fig_entropy.update_layout(height=420, width=900, showlegend=False)
fig_entropy.show()

radar_metrics = ['reward', 'cumulative_return', 'sharpe_ratio', 'action_entropy']
radar_df = cmp_df[['model'] + radar_metrics].copy()

for metric in radar_metrics:
    col = radar_df[metric].astype(float)
    valid = col.dropna()
    if len(valid) == 0:
        radar_df[metric] = np.nan
    elif abs(valid.max() - valid.min()) < 1e-12:
        radar_df[metric] = 0.5
    else:
        radar_df[metric] = (col - valid.min()) / (valid.max() - valid.min())

fig_radar = go.Figure()
for _, row in radar_df.iterrows():
    vals = [row[m] for m in radar_metrics]
    fig_radar.add_trace(
        go.Scatterpolar(
            r=vals + [vals[0]],
            theta=radar_metrics + [radar_metrics[0]],
            fill='toself',
            name=row['model'],
        )
    )

fig_radar.update_layout(
    title='Normalized Multi-Metric Comparison (higher is better)',
    polar=dict(radialaxis=dict(visible=True, range=[0, 1])),
    height=560,
    width=900,
    legend=dict(orientation='h', y=1.05, x=0),
)
fig_radar.show()

print("Comparison complete.")
print("Tip: if PPO-Native-Attn row is NaN, run Section 6.3 first.")

=== Model Comparison: DQN vs PPO-MLP vs PPO-Native-Attention ===
             model      reward  cumulative_return  sharpe_ratio  max_drawdown  \
0              DQN         NaN                NaN           NaN           NaN   
1          PPO-MLP  85742.2416                inf       41.0782           NaN   
2  PPO-Native-Attn  86325.3338                inf       41.4247           NaN   

   action_entropy  
0             NaN  
1          0.1592  
2         -0.0000  


Comparison complete.
Tip: if PPO-Native-Attn row is NaN, run Section 6.3 first.


In [21]:
# Auto-rank models with a simple risk-adjusted composite score
print("=== Auto Model Scoring and Winner Selection ===")

import numpy as np
import pandas as pd

def _safe_float(d, k, default=np.nan):
    try:
        return float(d.get(k, default)) if isinstance(d, dict) else float(default)
    except Exception:
        return float(default)

def _finite_or_default(x, default=0.0):
    try:
        x = float(x)
    except Exception:
        return float(default)
    return x if np.isfinite(x) else float(default)

def _calc_action_entropy(action_series):
    if action_series is None or len(action_series) == 0:
        return np.nan
    probs = action_series.value_counts(normalize=True).values
    return float(-(probs * np.log(probs + 1e-12)).sum())

def _estimate_max_drawdown_from_actions(action_df):
    if not isinstance(action_df, pd.DataFrame) or 'return' not in action_df.columns or len(action_df) == 0:
        return np.nan

    r = pd.to_numeric(action_df['return'], errors='coerce').replace([np.inf, -np.inf], np.nan).dropna()
    if r.empty:
        return np.nan

    # Build drawdown in log space to avoid overflow for long/high-growth series.
    # Clip only invalid downside (< -100%) to keep log1p stable.
    r = np.maximum(r.to_numpy(dtype=float), -0.999999)
    log_equity = np.cumsum(np.log1p(r))
    run_max = np.maximum.accumulate(log_equity)
    drawdowns = np.exp(log_equity - run_max) - 1.0
    return float(np.min(drawdowns)) if len(drawdowns) else np.nan

# Auto-backfill DQN test evaluation if DQN was trained but not evaluated in this session.
if ('test_eval' not in globals() or not isinstance(globals().get('test_eval'), dict)) and ('agent' in globals()) and ('test_env' in globals()):
    try:
        test_eval = evaluate_episode(agent, test_env, deterministic=True)
        if isinstance(test_eval, dict) and 'actions' in test_eval:
            actions_df = pd.DataFrame(test_eval['actions'])
        print("DQN eval auto-generated from current agent/test_env for ranking.")
    except Exception as e:
        print(f"DQN auto-eval skipped: {e}")

def _build_row(name, eval_dict=None, action_df=None, action_col=None):
    entropy = np.nan
    action_df_for_dd = action_df if isinstance(action_df, pd.DataFrame) else None

    if isinstance(action_df, pd.DataFrame) and action_col in action_df.columns:
        entropy = _calc_action_entropy(action_df[action_col])

    # Always prepare eval actions fallback for metrics like drawdown.
    eval_actions_df = None
    if isinstance(eval_dict, dict) and 'actions' in eval_dict:
        eval_actions_df = pd.DataFrame(eval_dict['actions'])
        if pd.isna(entropy) and 'action_name' in eval_actions_df.columns:
            entropy = _calc_action_entropy(eval_actions_df['action_name'])
        if action_df_for_dd is None or 'return' not in action_df_for_dd.columns:
            action_df_for_dd = eval_actions_df

    reward_raw = _safe_float(eval_dict, 'reward', np.nan)
    cumulative_return_raw = _safe_float(eval_dict, 'cumulative_return', np.nan)
    sharpe_ratio_raw = _safe_float(eval_dict, 'sharpe_ratio', np.nan)
    max_drawdown_raw = _safe_float(eval_dict, 'max_drawdown', np.nan)

    # Fallback: derive max drawdown from action-level returns when env stats are missing.
    if not np.isfinite(max_drawdown_raw):
        dd_fallback = _estimate_max_drawdown_from_actions(action_df_for_dd)
        if np.isfinite(dd_fallback):
            max_drawdown_raw = dd_fallback

    reward_score = _finite_or_default(reward_raw, 0.0)
    cumulative_return_score = _finite_or_default(cumulative_return_raw, 0.0)
    sharpe_ratio_score = _finite_or_default(sharpe_ratio_raw, 0.0)
    max_drawdown_score = _finite_or_default(max_drawdown_raw, 0.0)

    has_eval = isinstance(eval_dict, dict) and len(eval_dict) > 0
    composite_score = np.nan
    if has_eval:
        # Composite score (higher is better):
        #   + sharpe_ratio
        #   + 0.5 * cumulative_return
        #   - 0.5 * abs(max_drawdown)
        composite_score = sharpe_ratio_score + 0.5 * cumulative_return_score - 0.5 * abs(max_drawdown_score)

    return {
        'model': name,
        'has_eval': has_eval,
        'reward': reward_raw,
        'cumulative_return': cumulative_return_raw,
        'sharpe_ratio': sharpe_ratio_raw,
        'max_drawdown': max_drawdown_raw,
        'action_entropy': entropy,
        'composite_score': composite_score,
    }

rows = [
    _build_row('DQN', globals().get('test_eval'), globals().get('actions_df'), 'action_name'),
    _build_row('A2C', globals().get('ac_test_eval'), globals().get('prob_df'), 'chosen_action'),
    _build_row('PPO-MLP', globals().get('ppo_test_eval'), globals().get('ppo_prob_df'), 'chosen_action'),
    _build_row('PPO-Native-Attn', globals().get('ppo_attn_test_eval'), globals().get('attn_prob_df'), 'chosen_action'),
]

score_df = pd.DataFrame(rows)
print("\nModel availability:")
print(score_df[['model', 'has_eval', 'reward', 'cumulative_return', 'sharpe_ratio', 'max_drawdown']].round(4))

rank_df = score_df.dropna(subset=['composite_score']).copy()
rank_df = rank_df[np.isfinite(rank_df['composite_score'])]

if rank_df.empty:
    raise RuntimeError(
        "No evaluated model found in kernel state. Run at least one test-evaluation cell (A2C/PPO/PPO-Native or DQN eval section) first."
    )

rank_df = rank_df.sort_values('composite_score', ascending=False).reset_index(drop=True)
winner = rank_df.iloc[0]

print("\nRanking (best to worst):")
print(rank_df[['model', 'composite_score', 'sharpe_ratio', 'cumulative_return', 'max_drawdown', 'action_entropy']].round(4))

print("\nWinner:")
print(f"  Model: {winner['model']}")
print(f"  Composite score: {winner['composite_score']:.4f}")
print(f"  Sharpe: {winner['sharpe_ratio'] if pd.notna(winner['sharpe_ratio']) else float('nan'):.4f} | CumReturn: {winner['cumulative_return'] if pd.notna(winner['cumulative_return']) else float('nan'):.4f} | MaxDD: {winner['max_drawdown'] if pd.notna(winner['max_drawdown']) else float('nan'):.4f}")

print("\nDecision note:")
print("- Prefer the top composite score, then choose the lower drawdown model if scores are close.")

=== Auto Model Scoring and Winner Selection ===

Model availability:
             model  has_eval      reward  cumulative_return  sharpe_ratio  \
0              DQN      True  86325.3338                inf       41.4247   
1              A2C      True  66509.2075                inf       24.8716   
2          PPO-MLP      True  85742.2416                inf       41.0782   
3  PPO-Native-Attn      True  86325.3338                inf       41.4247   

   max_drawdown  
0           0.0  
1           0.0  
2           0.0  
3           0.0  

Ranking (best to worst):
             model  composite_score  sharpe_ratio  cumulative_return  \
0              DQN          41.4247       41.4247                inf   
1  PPO-Native-Attn          41.4247       41.4247                inf   
2          PPO-MLP          41.0782       41.0782                inf   
3              A2C          24.8716       24.8716                inf   

   max_drawdown  action_entropy  
0           0.0         -0.0000  


## 6.4.1 Auto Model Scoring and Winner

> Automatic ranking across trained models using a risk-adjusted composite score.

In [22]:
# Final checkpoint: verify all model weight artifacts were saved
from pathlib import Path

if 'MODEL_PATHS' not in globals():
    repo_root = Path.cwd().resolve()
    if not (repo_root / 'ml').exists():
        found_root = None
        for candidate in [repo_root] + list(repo_root.parents):
            if (candidate / 'ml').exists() and (candidate / 'data').exists():
                found_root = candidate
                break
        if found_root is None:
            raise FileNotFoundError("Could not locate project root containing 'ml' and 'data'.")
        repo_root = found_root

    model_dir = repo_root / 'output' / 'models'
    MODEL_PATHS = {
        'dqn': model_dir / 'dqn_finrl_agent.zip',
        'a2c': model_dir / 'a2c_finrl_agent.zip',
        'ppo': model_dir / 'ppo_finrl_agent.zip',
        'ppo_native_attention': model_dir / 'ppo_native_attention_finrl_agent.zip',
    }

print('=== Model Artifact Verification ===')
missing = []

for name, path in MODEL_PATHS.items():
    exists = path.exists()
    size_mb = path.stat().st_size / (1024 * 1024) if exists else 0.0
    status = 'OK' if exists else 'MISSING'
    try:
        display_path = path.relative_to(repo_root)
    except ValueError:
        display_path = path
    print(f"{name:>22}: {status:7} | {display_path} | {size_mb:.2f} MB")
    if not exists:
        missing.append(name)

if missing:
    print('\nMissing models:', ', '.join(missing))
    print('Run the corresponding training+save cells before deleting the original notebook.')
else:
    print('\nAll model artifacts are present. The split notebooks are ready to use standalone.')

=== Model Artifact Verification ===
                   dqn: OK      | output/models/dqn_finrl_agent.zip | 0.40 MB
                   a2c: OK      | output/models/a2c_finrl_agent.zip | 0.40 MB
                   ppo: OK      | output/models/ppo_finrl_agent.zip | 0.58 MB
  ppo_native_attention: OK      | output/models/ppo_native_attention_finrl_agent.zip | 1.03 MB

All model artifacts are present. The split notebooks are ready to use standalone.
